 # <center> Problem Set 6 (Finetuning MACE) <center>
<center> Spring 2025 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51 <center>
<center> Due: Monday, May 12, 2025 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

Before starting, make sure to **request a GPU**! For this PSET, a **T4 GPU** should be sufficient to complete all problems. If you like, you can first debug things on CPU, then use a T4 GPU. The last problem may require a A100 given memory requirements.

### Download required data

In [ ]:
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-mace/data/pfkfb3_automap_ligands.sdf
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-mace/data/pfkfb3_automap_protein.pdb
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-mace/data/1uao.pdb
# we also need the MACE weights, but these will be autocollected when we download the calculators.

In [ ]:
# do not modify!
!pip install rdkit tqdm mace-torch py3Dmol umap-learn numpy==2.0.0 openbabel-wheel torch-geometric
!pip install git+https://github.com/imagdau/aseMolec@main
# let's try using a version that lets us freeze certain layers
!git clone -b mace-freeze https://github.com/7radians/mace-freeze.git
!pip install ./mace-freeze

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors,Crippen
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
import itertools
from tqdm import tqdm
import mace
import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt
import py3Dmol
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact, IntSlider
from openbabel import pybel

import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.utils import shuffle

matplotlib.rcParams.update({'font.size': 15})
matplotlib.rc('lines', linewidth=3, color='g')
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams["xtick.major.size"] = 6
matplotlib.rcParams["ytick.major.size"] = 6
matplotlib.rcParams["ytick.major.width"] = 2
matplotlib.rcParams["xtick.major.width"] = 2
matplotlib.rcParams['text.usetex'] = False

In Problem Set 6, you'll explore working with a "foundational" machine learning interatomic potential (MLIP) model, MACE, which can be used to predict the forces acting upon or the energies of an atomic structure, with the following objectives:
* Visualizing conformers with `py3DMol` and their energies
* Querying energies and atom-level embeddings from MACE
* Comparing from-scratch vs. fine-tuning strategies for learning how to predict ΔG binding energy based on limited experimental data.
* Understanding limitations of these models

The MACE documentation highlights a lot of possible use cases, so I'd recommend reading through some of the docs to get an idea of what kinds of challenges MACE has addressed through feature availability (e.g. training different levels of theory, using cu-equivariance, MD simulations, etc); https://mace-docs.readthedocs.io/en/latest/guide/intro.html

# Part 1: Exploring conformer distribution and utilizing MACE to look at energies (25 points)

MACE is an equivariant neural network interatomic potential (NNIP) architecture, which researchers have used across a number of chemical domains to predict the energies and forces of various atomic systems. People have trained MACE on large swaths of different chemical spaces in an effort to offer "foundational" models tailored for problems in that space. The MACE model pretrained on the Materials Project data is referred to as MACE-MP-0; the MACE model trained on biomolecular systems (e.g. solvated amino acids, amino-acid ligand pairs, etc) as MACE-OFF. There are many versions and sizes of pretrained MACE models, which can be found in the MACE, MACE-OFF, and MACE-foundation libraries.


Here, we will use MACE to refer to the general strategy of using a NNIP or the model library/architecture itself, and MACE-MP-0 or MACE-OFF to refer to the pretrained models (in this pset, we'll stick with the `medium` sizes of both models).

Our goal for this PSET is a little extrapolative: pretrained MACE models are quite good at predicting simple electronic/quantum properties, so we've picked a more challenging prediction task of binding energy, $\Delta G_{bind}$ between a protein and ligand.
In an ideal world, we would calculate this term as follows:
$$\Delta G_{bind} = E(\text{protein and ligand bound}) - E(\text{protein and ligand unbound})$$


To simplify matters (hence making things more challenging for MACE), we will select only one protein system--PFKFB3--and use the few ligands reported for which we have both conformer information and experimental binding affinity, which has been converted into a ΔG for our use case. We will refrain from supplying the protein or solvent atomic coordinates to speed up prediction.



## Part 1.1: Visualize molecule conformers & datasets (7.5 points)
First, let's load our dataset with `pybel`, the Python library for openbabel. For this visualization, it'll be informative to view how each of the molecules fit into the protein binding pocket, so as an auxiliary, let's also load the pdb file.


In [ ]:
mols = [m for m in pybel.readfile("sdf", "pfkfb3_automap_ligands.sdf") if m is not None]
pdb = pybel.readfile("pdb", "pfkfb3_automap_protein.pdb")

We should first visualize our molecules (or conformers thereof); here is an example with `py3Dmol`:

In [ ]:
conformer1 = """
6
Conformer 1
C 0.000 0.000 0.000
H 0.000 0.000 1.089
H 1.026 0.000 -0.363
H -0.513 0.890 -0.363
H -0.513 -0.890 -0.363
H 0.513 0.890 -0.363
"""
# set up a Javascript viewer
view = py3Dmol.view(width=800, height=400)

# add a molecule by passing the object and giving it its type.
view.addModel(conformer1, "xyz")
# objects are 0-indexed; to set the style, we can specify the model we want to change,
# then the specific visualization styles we want.
view.setStyle({'model': 0}, {"stick": {}, "sphere": {"colorscheme": "Jmol"}})

# necessary for setting viewer
view.zoomTo()
view.show()

Task 1: Visualize the conformers using py3Dmol, alongside the protein structure for reference (set a lower opacity for the PDB structure only).
To get the "block" representation, you can use the `.write(format)` function of a pybel molecule object.


It is encouraged to try using a Jupyter slider to make it easy to visualize all the conformers, like so:

```
slider = IntSlider(min=0, max=len(mols)-1, step=1, description="Pose:")
interact(view_mol, idx=slider)
```
And all that is needed is to define a view_mol that takes in an `idx` keyword.

In [ ]:
########## Code ###########
# your code for view_mol here

# Commented to let solutions run
# slider = IntSlider(min=0, max=len(mols)-1, step=1, description="Pose:")
# interact(view_mol, idx=slider)
########## Code ###########

Task 2: Describe some of the visual differences you see amongst the conformers in this dataset. What features look preserved? What atoms of the pocket appear to be in interaction with the molecules? Do you see any possible correlations between observed molecular motifs and experimental ΔG?

A: Your Answer here

## Part 1.2: Compute energies of conformers with MACE (5 points)

Let's analyze the energies of these conformers. The MACE library offers a handy way with their `calculators` submodule to use a pretrained energetic model, and predict their energies. Let's try!

Task: use the mace-mp-0 and mace_off calculations to come up with energies for each of your conformers. Plot a scatterplot between each of the molecules and report any differences, for individual molecules and about the general distribution of the data.

Helpful hint: You may need to use `torch.set_default_dtype(torch.float64)` (for MACE-OFF) or `torch.set_default_dtype(torch.float32)` (for MACE-MP-0) to rectify any type errors that might arise from switching the two. This is because of a caching issue with the MACE library.

In [ ]:
########## Code ##########
from mace.calculators import mace_off, mace_mp
from ase import Atoms
def build_atom(conf):
    atoms = [a.GetSymbol() for a in conf.GetAtoms()]
    poss = [conf.GetConformer().GetAtomPosition(i) for i, _ in enumerate(conf.GetAtoms())]
    return Atoms(atoms, poss)

## Code to set up calculators and collect energies
mace_off_energies = []
mace_mp_energies = []








########## Code ##########

In [ ]:
########## Code ##########
# Code for scatterplot

########## Code ##########

A: Report your findings here

## Part 1.3: plotting atom-level descriptors/features as derived from MACE (12.5 points)

As you may recall from lecture and PSET 3, GNNs build atom-specific features over successive layers; while these are usually aggregated in some manner to predict a molecule-level property, we can nonetheless extract and utilize the atom-level features as possibly informative embeddings.

MACE also offers such atom embeddings, known as descriptors, from https://mace-docs.readthedocs.io/en/latest/guide/descriptors.html.

Task 1: Following the documentation above, let's compare the utility of the MACE-OFF vs.
 MACE-MP-0 descriptors. Collect the physical descriptors as generated by both MACE-OFF and MACE-MP-0 separately (using `invariants_only=True`). Average embeddings over all atoms per molecule (so you have one embedding per molecule), and try clustering them with UMAP (feel free to refer back to PSET 4 for help on this problem). Color the samples based on their experimental $\Delta G$ value.

In [ ]:
########## Code ##########


########## Code ##########


**Task 2**: answer the following questions:   
1) What do you observe in terms of the clustering captured by the UMAP embeddings relative to the $\Delta G$ values? Do certain clusters capture a subset of $\Delta G$ well? Why do you think this might be the case?

2) Does one model outperform the other based on the quality of clustering or separation? What information could be useful in aiding the model to perform better on out of distribution data given atomic features?


3) Based on the quality of separation, what do you think the performance of building a regressor from these embeddings will be?

A: Your answer here

# Part 2: Comparing different strategies for transfer learning and finetuning with MACE (40 points)

Now that we've explored some of the original model's capabilities, let's try leveraging what it has learned. Our goal for this problem set will be to explore the benefits of finetuning through two different means, compared to trying to train something given no other starting information.

We'll first train a MACE model from scratch. "From scratch" means that we do _not_ use the pretrained weights and instead have to try to learn the property directly. The authors of MACE refer to MACE as the architecture; and MACE-MP-0 or MACE-OFF to describe pretrained model weights. MACE is a 3D equivariant GNN architecture, so in theory you can repurpose it to learn any property where it makes sense to utilize 3D properties. Then, we'll finetune a pretrained MACE model on this property. Lastly, we'll try using the descriptors collected in Part 1.3 to train a simple regressor on the atom features to see if that aids the modeling task.

## Part 2.1: Make train/validation/test splits for training (5 points)

The MACE library offers a simple CLI to handle model training with simple configurations, which we'll use so as not to modify library functions. To use the CLI functionality, we'll need to create our splits and save these to disk.

Task 1: Create the splits on the ligand dataset. We will use the `ase` library which MACE relies on to process its data, and save our ligands in an `xyz` format. To simplify some of this process, we'll first save all the conformers in an XYZ file:

In [ ]:
from rdkit import Chem

suppl = Chem.SDMolSupplier("pfkfb3_automap_ligands.sdf", removeHs=False)
with open("all_ligands.xyz", "w") as out:
    # out.write(e0s)
    for mol in suppl:
        if mol is None:
            continue
        n = mol.GetNumAtoms()
        name = mol.GetProp("_Name")
        property_line = f'''Properties=species:S:1:pos:R:3 Comp=VC(3) name={name} r_exp_dg={mol.GetProp("r_exp_dg")} pbc="F F F"'''
        xyz_block = Chem.MolToXYZBlock(mol)
        xyz_block = xyz_block.split("\n")[2:] # drop first two lines to use our property line.
        xyz_block = "\n".join(xyz_block)
        out.write(f"{n}\n{property_line}\n{xyz_block}")

Now you can use `ase` with `all_ligands.xyz`. Use the read/write functions in `ase.io` [(documentation here)](https://wiki.fysik.dtu.dk/ase/ase/io/io.html) to create a train, validation, and test split. You can assume the data is shuffled for you; use a split of 70%-10%-20% t-v-t.

In [ ]:
########## Code ##########

########## Code ##########

## Part 2.2: Train MACE from scratch (10 points)
For the first experiment, let's try training a new MACE model without any pretraining.

Task 1: Set up a config file with the specifications in the PDF, then train using the provided utility wrapper we have provided you.

In [ ]:
%%writefile config_from_scratch.yml
# fill! remove this comment when done so that it does not get printed to the config file.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mace.cli.run_train import main as mace_run_train_main
import sys
import logging

def train_mace(config_file_path):
    logging.getLogger().handlers.clear()
    sys.argv = ["program", "--config", config_file_path]
    mace_run_train_main()

In [ ]:
train_mace("config_from_scratch.yml")

Task 2: Now that we have a trained model, let's take a coarse evaluation of performance. Use the utility to produce predictions on the training, validation, and test sets (we recommend naming output files by both split and training type, to help disambiguate from later problems!), and plot a scatterplot of predicted vs. experimental values. Report the R^2 and RMSE for each split.

For a helpful reference using the `aseMolec` library, see here:

In [ ]:
from mace.cli.eval_configs import main as mace_eval_configs_main
import sys

def eval_mace(configs, model, output):
    sys.argv = ["program", "--configs", configs, "--model", model, "--output", output]
    mace_eval_configs_main()


########### Code ###########
# Change the Nones, and repeat for train/val/test split
# eval_mace(configs=None, # This should be your input file
#           model=None, # Path to your (stage two) model weights
#           output=None # Desired name of output file.
# )
########### Code ###########



In [ ]:
########### Code ###########
from aseMolec import pltProps as pp
from ase.io import read
import matplotlib.pyplot as plt
from aseMolec import extAtoms as ea
import numpy as np
# plotting code w/ R^2 and RMSE here
########### Code ###########

## Part 2.3: Finetune MACE-OFF on the same dataset (5 points)
For our second experiment, we'll try finetuning the MACE-OFF medium model. Similar to Part 2.2, follow the steps to create a config file and train for 50 epochs. Then, create the scatterplots as you did for 2.2.

Bonus: you are welcome to try using `--freeze=n` to try freezing the first `n` layers of the model, to see if that buys you any performance. Full credit on this problem, though, would not require you to do so but you are encouraged to explore!

In [ ]:
%%writefile config_finetune.yml
# fill! remove this comment when done so that it does not get printed to the config file.

In [ ]:
train_mace("config_finetune.yml")

In [ ]:
########### Code ###########
# Collect predictions
########### Code ###########

In [ ]:
########### Code ###########
# plotting code w/ R^2 and RMSE here
########### Code ###########

## Part 2.4: Train mini GNN/MLP on MACE descriptors (20 points)
For our third experiment, we'll try a different strategy of learning from the pretrained model -- building a simple regressor to learn binding energy from the MACE descriptors we found in 1.3.

### Part 2.4.1 Preliminary questions (5 points)

Answer the following questions:   
1) What are the conceptual differences between finetuning and using these descriptors as our input?

2) What might be the benefits of designing a predictive model separate from the MACE architecture?

A: Your answer here

### Part 2.4.2 Build a mini GNN/MLP architecture to predict $\Delta G$ (10 points)

Let's build a simple MLP that can operate on the atom embeddings. Using your *non-averaged*, per-atom **MACE-OFF** embeddings from 1.3, build a train/validation/test split, DataLoaders (of batch size 4) for your data, and a simple GNN/MLP architecture as follows:  

1. (at least) one GCNConv layer between atoms that preserves the size of the embedding.
2. an MLP, composed of two linear layers with requisite nonlinearities, that operates on each embedding individually
3. a pooling step to aggregate per-atom embeddings into a single value.   


Feel free to refer back to the code you wrote in PSET 3 to help you with this.

Helpful hints:
1. Remember that for a GNN convolution, you'll need edge_indices. You can get these from the pybel molecule object with `.GetBond(idx)`.
2. Make sure that your descriptor and edge indices are consistent with the number of (heavy) atoms in the molecule. You may need to use `mol.DeleteHydrogens()` to help fix this.
3. We encourage you to use the PyTorch Geometric library as seen in PSET 3, not just for convolutions but also for the data batching with the `Data` object, which can automatically handle collations/slicing of batches for you. Take a look [here](https://pytorch-geometric.readthedocs.io/en/2.5.0/tutorial/create_dataset.html) for reference. You can return a `Data` object with the `x` and `edge_index` keywords set properly in your `Dataset.__getitem__` function.

In [ ]:
########### Code ###########
from torch_geometric.data import Data
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
torch.set_default_dtype(torch.float64) # since we are using MACE-OFF


# Set up your dataset and dataloaders, as well as your splits.

########### Code ###########

In [ ]:
########### Code ###########

class MoleculeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        ### Fill

        ### Fill

    def forward(self, data):
        ### Fill
        return ### Fill

########### Code ###########

In [ ]:
########### Code ###########
# Write a training loop here.
########### Code ###########

### Part 2.4.3: Plot scatterplot of predictions (5 points)

With the model trained in 2.4.2, generate predicted ΔG values and plot them against their experimental values.

In [ ]:
########### Code ###########
# Write a prediction and scatterplot here.
########### Code ###########

## Part 2.5: Overall evaluation and observations (5 points)
We just tried three different strategies for training! Report some of the differences you see in performance, any outliers or handicaps you observe in quality, and what you think overall the best strategy could be. Finally, as a conceptual question, if we had multiple protein-ligand systems that we wanted to try using the MACE architecture to train on, what would be your preferred strategy of learning binding energy?

A: Your Answer Here

# Part 3: Run MD simulations with the MACE potential (5 points)

As a final (fun) task, let's also explore how one can use a learned potential (e.g. MACE-OFF or MACE-MP-0) to compute molecular dynamics simulations. Typically, these are done with physics-based force fields which can be at times slow to run and therefore prohibitive to running large-scale simulations. Neural network potentials like MACE-OFF offer the opportunity to accelerate these simulations, though the sacrifice in accuracy can vary across different systems and configurations.

Let's try running a simulation! Since the full protein-ligand system is too large, we'll use a different protein (`1uao.pdb`) as an example. Following the instructions [here](https://mace-docs.readthedocs.io/en/latest/guide/ase.html. ), set up a Langevin simulation to run for 500 timesteps with an interval of 25.

In [ ]:
########### Code ###########
# generate your trajectory here, and save to md_protein.xyz


########### Code ###########

In [ ]:
# Demonstrate you were able to run the simulation properly:
import mdtraj as md
traj = md.load('md_protein.xyz', top='1uao.pdb')
print(len(traj)) # should be 21 frames

In [ ]:
# Code to visualize trajectory -- may be buggy
import py3Dmol
with open('md_protein.xyz', 'r') as f:
    xyz_text = f.read()


view = py3Dmol.view(width=400, height=400)
view.addModel(xyz_text, "xyz", {'vibrate': {'frames':10,'amplitude':1}})

view.setStyle({'sphere':{'scale':0.30},'stick':{'radius':0.25}})
view.setBackgroundColor('0xeeeeee')
view.animate({'loop': 'backAndForth'})
view.zoomTo()
view.show()


Task: Analyze what changes you see occurring, if any, within the trajectory. If the _animation_ does not load properly, you can download the trajectory locally to visualize it with a tool like PyMol or a local jupyter notebook to visualize the trajectory. Based on your findings from earlier stages of the homework, how accurate would you expect this simulation to be? What changes do you think you could make to help improve the quality of the simulation?

A: Your answer here